# Phase 5b — Nested Cluster CV × Multi-Seed Ablation

Phase 5 returned an ambiguous verdict: on the single held-out split, topology
gained **+0.102 PR-AUC on validation** but **−0.003 on test**. With 144 positive
residues in that test split, one chain can move PR-AUC further than the gap
itself. That is not a result in either direction, so this notebook replaces the
single split with a design that can actually answer the question.

### What changes

1. **Nested cluster cross-validation.** 5 outer folds over the 21 sequence
   clusters, so *every* chain is held out exactly once and the test evaluation
   covers all 10,087 residues instead of 1,166. Inner clusters are split again
   for early stopping, so no test fold ever influences model selection.
2. **3 seeds per configuration.** 5 folds × 3 seeds × 3 arms = 45 training runs,
   giving 15 paired observations per arm comparison.
3. **Paired statistics.** The arms see identical folds and identical seeds, so
   differences are paired and a Wilcoxon signed-rank test is valid. Reporting a
   mean without the paired spread is what produced the ambiguity last time.

**The architecture is unchanged from Phase 5 on purpose.** Overfitting was severe
(train PR-AUC 0.97 vs val 0.43) and capacity should be tuned — but changing the
model and the evaluation protocol at once would confound the comparison. One
variable at a time; capacity is the next experiment, not this one.

## Cell 1 — Setup, model definitions (identical to Phase 5)

In [ ]:
!pip install -q torch-geometric==2.6.1

In [ ]:
import copy, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import GroupKFold


def locate(fname):
    c = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            c.extend(root.rglob(fname))
    if not c:
        raise FileNotFoundError(fname)
    return max(c, key=lambda p: p.stat().st_size)


blob = torch.load(locate("phase4_splits.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
TOPO_NAMES = blob["topo_names"]
TOPO_DIM = len(TOPO_NAMES)
N_SEQ = int(dataset[0].x.size(1)) - TOPO_DIM
BURIAL_IDX = [TOPO_NAMES.index(n) for n in blob["burial_names"]]
POS_WEIGHT = float(blob["pos_weight"])

GPU_OK = False
if torch.cuda.is_available():
    try:
        _p = torch.randn(256, 256, device="cuda"); (_p @ _p).sum().item()
        torch.cuda.synchronize(); GPU_OK = True
    except RuntimeError:
        pass
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"device {DEVICE} | seq {N_SEQ} + topo {TOPO_DIM} | pos_weight {POS_WEIGHT:.2f}")
print(f"clusters {len({d.cluster for d in dataset})} | chains {len(dataset)}")

In [ ]:
class EGCL(nn.Module):
    def __init__(self, h_dim, edge_attr_dim=2, act=nn.SiLU, update_coords=True):
        super().__init__()
        self.update_coords = update_coords
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * h_dim + 1 + edge_attr_dim, h_dim), act(),
            nn.Linear(h_dim, h_dim), act())
        self.node_mlp = nn.Sequential(nn.Linear(2 * h_dim, h_dim), act(),
                                      nn.Linear(h_dim, h_dim))
        self.att = nn.Sequential(nn.Linear(h_dim, 1), nn.Sigmoid())
        if update_coords:
            self.coord_mlp = nn.Sequential(nn.Linear(h_dim, h_dim), act(),
                                           nn.Linear(h_dim, 1, bias=False))
            nn.init.xavier_uniform_(self.coord_mlp[-1].weight, gain=0.001)
        self.norm = nn.LayerNorm(h_dim)

    def forward(self, h, x, edge_index, edge_attr):
        src, dst = edge_index
        rel = x[src] - x[dst]
        d2 = (rel ** 2).sum(-1, keepdim=True)
        m = self.edge_mlp(torch.cat([h[src], h[dst], d2, edge_attr], dim=-1))
        m = m * self.att(m)
        agg = torch.zeros_like(h).index_add_(0, dst, m)
        deg = torch.zeros(h.size(0), 1, device=h.device).index_add_(
            0, dst, torch.ones_like(d2)).clamp(min=1)
        h = self.norm(h + self.node_mlp(torch.cat([h, agg / deg], dim=-1)))
        if self.update_coords:
            w = self.coord_mlp(m) * rel / (d2.sqrt() + 1.0)
            x = x + torch.zeros_like(x).index_add_(0, dst, w) / deg
        return h, x


class TopoFusion(nn.Module):
    def __init__(self, h_dim, topo_dim):
        super().__init__()
        self.proj = nn.Linear(topo_dim, h_dim)
        self.gate = nn.Sequential(nn.Linear(h_dim + topo_dim, h_dim), nn.Sigmoid())

    def forward(self, h, t):
        g = self.gate(torch.cat([h, t], dim=-1))
        self.last_gate = g.detach()
        return h + g * self.proj(t)


class TopologicalEGNN(nn.Module):
    def __init__(self, n_seq, topo_dim, h_dim=128, n_layers=4, edge_attr_dim=2, dropout=0.1):
        super().__init__()
        self.n_seq = n_seq
        self.embed = nn.Sequential(nn.Linear(n_seq + topo_dim, h_dim), nn.SiLU(),
                                   nn.Linear(h_dim, h_dim))
        self.layers = nn.ModuleList([EGCL(h_dim, edge_attr_dim, update_coords=(i < n_layers - 1))
                                     for i in range(n_layers)])
        self.fusions = nn.ModuleList([TopoFusion(h_dim, topo_dim) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(h_dim, h_dim // 2), nn.SiLU(),
                                  nn.Dropout(dropout), nn.Linear(h_dim // 2, 1))

    def forward(self, data):
        h = self.embed(data.x)
        pos = data.pos
        topo = data.x[:, self.n_seq:]
        for layer, fuse in zip(self.layers, self.fusions):
            h, pos = layer(h, pos, data.edge_index, data.edge_attr)
            h = fuse(h, topo)
        return self.head(h).squeeze(-1)

## Cell 2 — Ablation masking

Each arm zeroes a different slice of the topological block. Architecture and
parameter count stay identical across arms, so the only thing that varies is the
information available — which is what makes the comparison interpretable.

In [ ]:
ORIG_X = [d.x.clone() for d in dataset]


def apply_arm(arm: str):
    for d, x0 in zip(dataset, ORIG_X):
        t = x0[:, N_SEQ:].clone()
        if arm == "seq-only":
            t.zero_()
        elif arm == "burial-only":
            keep = torch.zeros_like(t)
            keep[:, BURIAL_IDX] = t[:, BURIAL_IDX]
            t = keep
        d.x = torch.cat([x0[:, :N_SEQ], t], dim=1)


def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    ps, ys = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(batch)
            loss = crit(logits, batch.y)
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy())
        ys.append(batch.y.detach().cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    return {"roc": roc_auc_score(y, p) if y.sum() else float("nan"),
            "pr": average_precision_score(y, p) if y.sum() else float("nan"),
            "p": p, "y": y}


def best_mcc(p, y):
    best = -1.0
    for t in np.linspace(0.05, 0.95, 91):
        pred = (p >= t).astype(int)
        if pred.sum():
            best = max(best, matthews_corrcoef(y, pred))
    return best


def fit(train_ds, val_ds, seed, epochs=120, patience=25):
    torch.manual_seed(seed); np.random.seed(seed)
    model = TopologicalEGNN(N_SEQ, TOPO_DIM).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(train_ds, batch_size=8, shuffle=True)
    vl = DataLoader(val_ds, batch_size=8)
    best, state, bad = -1.0, None, 0
    for _ in range(epochs):
        run_epoch(model, tl, opt)
        v = run_epoch(model, vl)
        sched.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(state)
    return model, best

## Cell 3 — Nested CV × seeds

Outer loop holds out whole clusters for test; the inner split carves validation
clusters out of the remaining training clusters. No test fold ever touches early
stopping, so the reported numbers are genuinely out-of-sample.

In [ ]:
ARMS = ["seq-only", "burial-only", "full"]
SEEDS = [0, 1, 2]
N_FOLDS = 5

clusters = np.array([d.cluster for d in dataset])
y_dummy = np.zeros(len(dataset))
folds = list(GroupKFold(n_splits=N_FOLDS).split(np.zeros(len(dataset)), y_dummy, clusters))

records = []
t0 = time.time()
for arm in ARMS:
    apply_arm(arm)
    for fi, (tr_idx, te_idx) in enumerate(folds):
        # inner split: hold out ~20% of TRAIN clusters for early stopping
        tr_clusters = sorted({clusters[i] for i in tr_idx})
        n_val = max(1, len(tr_clusters) // 5)
        for seed in SEEDS:
            rng = np.random.RandomState(1000 + seed)
            val_c = set(rng.choice(tr_clusters, size=n_val, replace=False))
            tr_ds = [dataset[i] for i in tr_idx if clusters[i] not in val_c]
            va_ds = [dataset[i] for i in tr_idx if clusters[i] in val_c]
            te_ds = [dataset[i] for i in te_idx]
            if not va_ds or not tr_ds:
                continue

            model, val_pr = fit(tr_ds, va_ds, seed)
            te = run_epoch(model, DataLoader(te_ds, batch_size=8))
            records.append({"arm": arm, "fold": fi, "seed": seed,
                            "val_pr": val_pr, "test_pr": te["pr"],
                            "test_roc": te["roc"], "test_mcc": best_mcc(te["p"], te["y"]),
                            "n_test_pos": int(te["y"].sum()), "n_test": int(len(te["y"]))})
            print(f"  {arm:<12} fold {fi} seed {seed} | val PR {val_pr:.3f} "
                  f"| test PR {te['pr']:.3f} ROC {te['roc']:.3f} "
                  f"(n={records[-1]['n_test']}, pos={records[-1]['n_test_pos']})")

for d, x0 in zip(dataset, ORIG_X):
    d.x = x0
print(f"\n{len(records)} runs in {(time.time() - t0) / 60:.1f} min")

## Cell 4 — Paired analysis

In [ ]:
from scipy.stats import wilcoxon

def get(arm, metric):
    return np.array([r[metric] for r in sorted(
        [x for x in records if x["arm"] == arm], key=lambda x: (x["fold"], x["seed"]))])

print(f"{'arm':<14}{'test PR-AUC':>18}{'test ROC-AUC':>18}{'test MCC':>16}")
print("-" * 66)
for arm in ARMS:
    pr, roc, mcc = get(arm, "test_pr"), get(arm, "test_roc"), get(arm, "test_mcc")
    print(f"{arm:<14}{pr.mean():>10.3f} +-{pr.std():.3f}"
          f"{roc.mean():>11.3f} +-{roc.std():.3f}{mcc.mean():>9.3f} +-{mcc.std():.3f}")

print("\n" + "=" * 66)
print("PAIRED COMPARISONS (same fold, same seed)")
for a, b in (("full", "burial-only"), ("full", "seq-only"), ("burial-only", "seq-only")):
    for metric in ("test_pr", "test_roc"):
        d = get(a, metric) - get(b, metric)
        try:
            stat, p = wilcoxon(d)
        except ValueError:
            p = float("nan")
        print(f"  {a} - {b:<14} {metric:<9}: {d.mean():+.4f} +-{d.std():.4f}  "
              f"{(d > 0).sum()}/{len(d)} wins  Wilcoxon p={p:.4f}")

out = Path("/kaggle/working")
(out / "phase5b_cv_results.json").write_text(json.dumps(records, indent=2, default=float))
print(f"\nsaved -> {out / 'phase5b_cv_results.json'}")

## Cell 5 — Plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].boxplot([get(a, "test_pr") for a in ARMS], labels=ARMS)
for i, a in enumerate(ARMS):
    v = get(a, "test_pr")
    ax[0].scatter(np.full(len(v), i + 1) + np.linspace(-.08, .08, len(v)), v, s=14, alpha=.7)
ax[0].set_ylabel("test PR-AUC"); ax[0].set_title("nested cluster CV x 3 seeds")

d = get("full", "test_pr") - get("burial-only", "test_pr")
ax[1].axhline(0, color="k", lw=1)
ax[1].bar(range(len(d)), d)
ax[1].set_title(f"paired delta: full - burial-only  (mean {d.mean():+.4f})")
ax[1].set_xlabel("fold x seed"); ax[1].set_ylabel("PR-AUC difference")
plt.tight_layout(); plt.show()

---

## How to read this

The paired `full − burial-only` line is the verdict, and it now rests on 15
paired observations covering every chain rather than one 7-chain split.

- **Mean positive, most pairs winning, Wilcoxon p < 0.05** → topology contributes
  beyond burial. Report the mean ± std and the p-value, never a single fold.
- **Mean near zero or p large** → on this dataset, at this scale, persistent
  homology adds nothing beyond density/burial. That is a legitimate,
  publishable finding, and the residualisation evidence from Phase 2b still
  stands as a feature-level result — it simply means the EGNN cannot exploit it
  at 41 chains.

Either way the number is now honest. **The dominant limitation is no longer the
protocol — it is that 21 clusters is a small sample.** The BioLiP scale-up is what
would move this from "suggestive" to "demonstrated", and that is the single
highest-value next step for the project.